# Sealed gain on real data: EvalPlus (HumanEval+ and MBPP+)

Companion experiment to *Bits Saved* (journal version). Each task is a real Python function `f` from EvalPlus.
The agent has observed calls `f(args) == output`; it keeps 3 of them exposed and 8 sealed (the audit); 8 further
calls are the future it should predict. Candidate documents are the task description (SPEC), the reference
implementation (CODE), six further calls (EXAMPLES), two single-site mutants of CODE that change some outputs (WRONG),
a reordered copy of the exposed calls (ECHO), implementations of two other problems (DISTRACTOR), CODE's tokens in
random order (SHUFFLED), random base64 (NOISE) and generic prose (BOILER).

The tasks were built by `evalplus_tasks.py` (shipped alongside) from the EvalPlus release files; every output was
computed by running the reference implementation, so the ground truth is exact. The ranking signal is sealed gain
G (lambda = 0), fixed in advance from the synthetic study; nothing is tuned on these tasks.

**Run:** Runtime -> Change runtime type -> GPU (T4 or better). Upload `evalplus_tasks.json.gz` when asked (or put it
next to the notebook). Set `MODE = "FULL"` and run all. At the end, download `evalplus_results.zip`.

In [ ]:
# ---- 0.1  Setup ----
import sys, os, json, gzip, time, gc, re, math
from collections import defaultdict, Counter
IN_COLAB = "google.colab" in sys.modules
MODE = os.environ.get("EP_MODE", "FULL")        # "FULL" (GPU, real models) | "TEST" (CPU smoke test with a stand-in scorer)
CFG = dict(
    models=["Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-Coder-7B-Instruct"] if MODE == "FULL" else ["MOCK"],
    n_tasks=None if MODE == "FULL" else 12,       # None = all tasks
    judge=True, max_batch_tokens=12000, n_boot=2000, dose_pools=200, seed=0,
)
TASKS_PATH = "evalplus_tasks.json.gz"
OUT = os.path.abspath(os.environ.get("EP_RESULTS", "evalplus_results")); os.makedirs(OUT, exist_ok=True)
if IN_COLAB and not os.path.exists(TASKS_PATH):
    from google.colab import files
    print("Upload evalplus_tasks.json.gz"); files.upload()
print("MODE", MODE, "| models", CFG["models"], "| results ->", OUT)

MODE FULL | models ['Qwen/Qwen2.5-7B-Instruct', 'Qwen/Qwen2.5-Coder-7B-Instruct'] | results -> /content/evalplus_results


In [ ]:
# ---- 0.2  Imports and helpers ----
import numpy as np, pandas as pd
from scipy import stats
import matplotlib
if not IN_COLAB: matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"pdf.fonttype": 42, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

def boot_ci(x, n=None, alpha=0.05, seed=0):
    x = np.asarray([v for v in x if np.isfinite(v)], float)
    if len(x) == 0: return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed); n = n or CFG["n_boot"]
    bs = rng.choice(x, size=(n, len(x)), replace=True).mean(1)
    return (float(x.mean()), float(np.quantile(bs, alpha / 2)), float(np.quantile(bs, 1 - alpha / 2)))

def wilcoxon_p(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    try: return float(stats.wilcoxon(a, b).pvalue) if len(a) >= 5 and not np.allclose(a, b) else np.nan
    except ValueError: return np.nan

def fmt_ci(t, p=2): return f"{t[0]:.{p}f} [{t[1]:.{p}f}, {t[2]:.{p}f}]"

In [ ]:
# ---- 1  Scorers (same implementation as the synthetic study's notebook) ----
class Seg:
    __slots__ = ("name", "ids", "tags", "score")
    def __init__(self, name, ids, tags=None, score=False):
        self.name, self.ids, self.score = name, list(ids), score
        self.tags = np.zeros(len(self.ids), int) if tags is None else np.asarray(tags, int)

class BaseScorer:
    def __init__(self): self._tok_cache = {}
    def ids(self, text):
        if text not in self._tok_cache: self._tok_cache[text] = self._encode(text)
        return self._tok_cache[text]
    def seg_text(self, name, text, score=False): return Seg(name, self.ids(text), None, score)
    def seg_line(self, name, pieces, score=True):
        ids, tags = [], []
        for text, tag in pieces:
            t = self.ids(text); ids += t; tags += [tag] * len(t)
        return Seg(name, ids, tags, score)
    def score(self, seqs):
        flat = []
        for segs in seqs:
            ids, spans = [], []
            for sg in segs:
                s = len(ids); ids += sg.ids; spans.append((sg, s, len(ids)))
            flat.append((ids, spans))
        lp_all, ok_all = self._token_logprobs([f[0] for f in flat],
                                              [[p for sg, s, e in f[1] if sg.score for p in range(max(s, 1), e)] for f in flat])
        out = []
        for (ids, spans), lp, ok in zip(flat, lp_all, ok_all):
            res, j = {}, 0
            for sg, s, e in spans:
                if not sg.score: continue
                n = max(0, e - max(s, 1)); l, o = lp[j:j + n], ok[j:j + n]; j += n
                tags = sg.tags[len(sg.ids) - n:]
                spans_ok = [o[tags == t].all() for t in np.unique(tags[tags > 0])]
                res[sg.name] = dict(bits=float(-l.sum() / np.log(2)), n=n, attr_n=len(spans_ok), attr_correct=int(np.sum(spans_ok)))
            out.append(res)
        return out
    def embed(self, texts): return self._embed_ids([self.ids(t) for t in texts])
    def judge(self, prompts): return self._judge_texts(prompts)

class HFScorer(BaseScorer):
    def __init__(self, name, max_batch_tokens=12000):
        super().__init__()
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.torch, self.name = torch, name
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = (torch.bfloat16 if self.device == "cuda" and torch.cuda.is_bf16_supported() and torch.cuda.get_device_capability()[0] >= 8
                 else torch.float16 if self.device == "cuda" else torch.float32)
        self.tok = AutoTokenizer.from_pretrained(name)
        self.model = self._load(AutoModelForCausalLM, name, dtype)
        self.pad = self.tok.pad_token_id if self.tok.pad_token_id is not None else self.tok.eos_token_id
        self.mbt = max_batch_tokens
        self.yes_id = self.tok.encode("Yes", add_special_tokens=False)[0]; self.no_id = self.tok.encode("No", add_special_tokens=False)[0]
        test = self.score([[self.seg_text("a", "Notes:\n"), self.seg_text("b", "def f(x):\n    return x + 1", True)]])[0]["b"]["bits"]
        if not np.isfinite(test) and dtype != torch.float32:
            print("  non-finite logprobs -> reloading in float32"); del self.model; self.free_cuda()
            self.model = self._load(AutoModelForCausalLM, name, torch.float32)
        print(f"  loaded {name} on {self.device} ({next(self.model.parameters()).dtype})")
    def _load(self, cls, name, dtype):
        try: m = cls.from_pretrained(name, dtype=dtype)
        except TypeError: m = cls.from_pretrained(name, torch_dtype=dtype)
        return m.to(self.device).eval()
    def _encode(self, text): return self.tok(text, add_special_tokens=False).input_ids
    def free_cuda(self):
        gc.collect()
        if self.torch.cuda.is_available(): self.torch.cuda.empty_cache()
    def _parts(self):
        m = self.model
        return getattr(m, m.base_model_prefix), m.get_output_embeddings()
    def _batches(self, lens):
        order = sorted(range(len(lens)), key=lambda i: -lens[i]); b = []
        for i in order:
            if b and lens[b[0]] * (len(b) + 1) > self.mbt: yield b; b = []
            b.append(i)
        if b: yield b
    def _forward_hidden(self, id_lists):
        T = max(len(x) for x in id_lists)
        inp = self.torch.full((len(id_lists), T), self.pad, dtype=self.torch.long)
        att = self.torch.zeros((len(id_lists), T), dtype=self.torch.long)
        for r, x in enumerate(id_lists): inp[r, :len(x)] = self.torch.tensor(x); att[r, :len(x)] = 1
        backbone, _ = self._parts()
        return backbone(input_ids=inp.to(self.device), attention_mask=att.to(self.device), use_cache=False).last_hidden_state
    def _token_logprobs(self, id_lists, pos_lists):
        torch = self.torch; lp_out, ok_out = [None] * len(id_lists), [None] * len(id_lists)
        _, head = self._parts()
        with torch.inference_mode():
            for b in self._batches([len(x) for x in id_lists]):
                H = self._forward_hidden([id_lists[i] for i in b])
                rows, cols, tgts = [], [], []
                for r, i in enumerate(b):
                    p = pos_lists[i]; rows += [r] * len(p); cols += [q - 1 for q in p]; tgts += [id_lists[i][q] for q in p]
                if not rows:
                    for i in b: lp_out[i], ok_out[i] = np.zeros(0), np.zeros(0, bool)
                    continue
                hs = H[torch.tensor(rows, device=self.device), torch.tensor(cols, device=self.device)]
                tg = torch.tensor(tgts, device=self.device); lps, oks = [], []
                for c in range(0, len(tg), 1024):
                    lg = head(hs[c:c + 1024]).float(); ls = torch.log_softmax(lg, -1)
                    lps.append(ls.gather(1, tg[c:c + 1024, None])[:, 0]); oks.append(lg.argmax(-1) == tg[c:c + 1024])
                lp = torch.cat(lps); ok = torch.cat(oks)
                if not torch.isfinite(lp).all():
                    if next(self.model.parameters()).dtype == torch.float32: raise FloatingPointError("non-finite log-probs in float32")
                    print("  non-finite log-probs in half precision -> converting model to float32 and re-scoring")
                    self.model = self.model.float(); self.free_cuda()
                    return self._token_logprobs(id_lists, pos_lists)
                lp, ok = lp.cpu().numpy(), ok.cpu().numpy(); j = 0
                for i in b:
                    n = len(pos_lists[i]); lp_out[i], ok_out[i] = lp[j:j + n], ok[j:j + n]; j += n
        return lp_out, ok_out
    def _embed_ids(self, id_lists):
        out = [None] * len(id_lists)
        with self.torch.inference_mode():
            for b in self._batches([len(x) for x in id_lists]):
                H = self._forward_hidden([id_lists[i] for i in b]).float()
                for r, i in enumerate(b): out[i] = H[r, :len(id_lists[i])].mean(0).cpu().numpy()
        E = np.stack(out); return E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)
    def _judge_texts(self, prompts):
        texts = [self.tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True) for p in prompts]
        ids = [self._encode(t) for t in texts]; out = np.zeros(len(ids))
        _, head = self._parts()
        with self.torch.inference_mode():
            for b in self._batches([len(x) for x in ids]):
                H = self._forward_hidden([ids[i] for i in b])
                last = H[self.torch.arange(len(b), device=self.device), self.torch.tensor([len(ids[i]) - 1 for i in b], device=self.device)]
                lg = head(last).float()
                for r, i in enumerate(b): out[i] = float(lg[r, self.yes_id] - lg[r, self.no_id])
        return out
    def close(self): del self.model; self.free_cuda()

class MockScorer(BaseScorer):
    """CPU stand-in for smoke tests only (cache-bigram LM over characters classes). Not a real result."""
    def __init__(self, name="MOCK"): super().__init__(); self.name = name; self.vocab = {}
    def _encode(self, text):
        return [self.vocab.setdefault(t, len(self.vocab)) for t in re.findall(r"\s?[A-Za-z_]+|\s?\d+|\s?[^A-Za-z_\d\s]|\n", text)]
    def _token_logprobs(self, id_lists, pos_lists):
        lps, oks = [], []
        for ids, pos in zip(id_lists, pos_lists):
            big, uni, need, lp, ok = defaultdict(Counter), Counter(), set(pos), [], []
            for q in range(1, len(ids)):
                prev, t = ids[q - 1], ids[q]
                if q in need:
                    pb = big[prev][t] / (sum(big[prev].values()) + 1) if big[prev] else 0.0
                    pu = uni[t] / (sum(uni.values()) + 1)
                    lp.append(np.log(0.75 * pb + 0.2 * pu + 0.05 * 1e-5 + 1e-9))
                    ok.append(bool(big[prev]) and big[prev].most_common(1)[0][0] == t)
                big[prev][t] += 1; uni[t] += 1
            lps.append(np.array(lp)); oks.append(np.array(ok, bool))
        return lps, oks
    def _embed_ids(self, id_lists):
        E = np.zeros((len(id_lists), 256))
        for r, ids in enumerate(id_lists):
            for t in ids: E[r, (t * 2654435761) % 256] += 1
        return E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)
    def _judge_texts(self, prompts): return np.array([len(p) % 7 / 7.0 for p in prompts])
    def close(self): pass

def load_scorer(name):
    return MockScorer(name) if name.startswith("MOCK") else HFScorer(name, CFG["max_batch_tokens"])

In [ ]:
# ---- 2  Tasks ----
with gzip.open(TASKS_PATH, "rt") as fh: TASKS = json.load(fh)
for t in TASKS:
    for k in ("exp", "aud", "fut"): t[k] = [[tuple(p) for p in line] for line in t[k]]
if CFG["n_tasks"]: TASKS = TASKS[:CFG["n_tasks"]]
USEFUL_TYPES = ["SPEC", "CODE", "EXAMPLES"]
CONTROL_TYPES = ["WRONG", "ECHO", "DISTRACTOR", "SHUFFLED", "NOISE", "BOILER"]
dtype_of = lambda key: key.split("_")[0]
line_text = lambda pieces: "".join(p for p, _ in pieces)
n_he = sum(t["task_id"].startswith("HumanEval") for t in TASKS)
print(f"{len(TASKS)} tasks ({n_he} HumanEval+, {len(TASKS) - n_he} MBPP+), {len(TASKS[0]['docs'])} candidates each")

266 tasks (109 HumanEval+, 157 MBPP+), 11 candidates each


In [ ]:
# ---- 3  Evaluation engine: the same layout as the synthetic study ----
HEADER = "Notes:\n\n"
JUDGE_TMPL = ("An agent is trying to understand an unknown Python function f. These are the calls it has observed:\n{hist}\n"
              "Candidate document:\n{doc}\n\nWould reading this candidate document help the agent understand its observations and "
              "predict the outputs of future calls to f? Answer with a single word, Yes or No.")

class TaskEval:
    """Layout per pass: HEADER, 'Observation log:' + exposed calls (context C), then 'New document:' + D (cost S),
    then 'More observations:' + the 8 audit calls (sealed gain G), then the 8 future calls (utility U, which therefore
    also conditions on the audit). pos='front' puts D before the whole log, for the unsealed gain."""
    def __init__(self, task, sc):
        self.t, self.sc, self.cache, self.emb = task, sc, {}, {}
        self.docs = task["docs"]
        self.hist_text = "".join(line_text(l) for l in task["exp"] + task["aud"])
        self._exp = [sc.seg_line(f"e{i}", l) for i, l in enumerate(task["exp"])]
        self._rest = ([sc.seg_line(f"a{i}", ([("More observations:\n", 0)] if i == 0 else []) + list(l)) for i, l in enumerate(task["aud"])] +
                      [sc.seg_line(f"q{i}", l) for i, l in enumerate(task["fut"])])
    def seq(self, cand, pos):
        sc = self.sc
        segs = [sc.seg_text("hdr", HEADER)]
        dseg = ([Seg("H", sc.ids("New document:\n"), None, True), Seg("D", sc.ids(self.docs[cand] + "\n\n"), None, True)]
                if cand is not None else [])
        log = [sc.seg_text("loghdr", "Observation log:\n")] + self._exp
        if pos == "front": return segs + dseg + log + [sc.seg_text("gap", "\n")] + self._rest
        return segs + log + [sc.seg_text("gap", "\n")] + dseg + self._rest
    def run(self, reqs):
        todo = [r for r in dict.fromkeys(reqs) if r not in self.cache]
        if not todo: return
        for (cand, pos), res in zip(todo, self.sc.score([self.seq(c, p) for c, p in todo])):
            g = lambda p: [v for k, v in res.items() if k[0] == p and k[1:].isdigit()]
            self.cache[(cand, pos)] = dict(
                S=res["D"]["bits"] + res["H"]["bits"] if "D" in res else 0.0, S_doc=res["D"]["bits"] if "D" in res else 0.0,
                n_D=res["D"]["n"] if "D" in res else 0, exp=sum(v["bits"] for v in g("e")),
                aud=sum(v["bits"] for v in g("a")), q=sum(v["bits"] for v in g("q")), q_n=sum(v["n"] for v in g("q")),
                q_acc=sum(v["attr_correct"] for v in g("q")) / max(1, sum(v["attr_n"] for v in g("q"))))
    def signals(self, rng):
        cands = list(self.docs)
        self.run([(None, "base")] + [(c, "seal") for c in cands] + [(c, "front") for c in cands])
        E = self.sc.embed([self.docs[c] for c in cands] + [self.hist_text]); ec, eh = E[:-1], E[-1]
        J = (self.sc.judge([JUDGE_TMPL.format(hist=self.hist_text, doc=self.docs[c]) for c in cands]) if CFG["judge"]
             else np.full(len(cands), np.nan))
        b = self.cache[(None, "base")]; rows = []
        for i, c in enumerate(cands):
            d, fr = self.cache[(c, "seal")], self.cache[(c, "front")]
            G = b["aud"] - d["aud"]
            rows.append(dict(task=self.t["task_id"], key=c, type=dtype_of(c), useful=dtype_of(c) in USEFUL_TYPES,
                             S=d["S"], G=G, G_un=(b["exp"] + b["aud"]) - (fr["exp"] + fr["aud"]), U=b["q"] - d["q"],
                             U_acc=d["q_acc"] - b["q_acc"], CP1=G - d["S"], ratio=G / max(d["S"], 1e-6),
                             surprise=d["S_doc"] / max(d["n_D"], 1), low_ppl=-d["S_doc"] / max(d["n_D"], 1),
                             relevance=float(eh @ ec[i]), judge=float(J[i]), random=float(rng.random())))
        return rows

In [ ]:
# ---- 4  Run every observer ----
ALL = []
for model in CFG["models"]:
    print("Observer:", model); sc = load_scorer(model); t0 = time.time(); rows = []
    for i, task in enumerate(TASKS):
        rows += TaskEval(task, sc).signals(np.random.default_rng(CFG["seed"] + i))
        if (i + 1) % 25 == 0: print(f"  {i + 1}/{len(TASKS)} tasks ({time.time() - t0:.0f}s)")
    df = pd.DataFrame(rows).assign(model=model.split("/")[-1]); ALL.append(df)
    df.to_csv(os.path.join(OUT, f"scores_{model.split('/')[-1]}.csv"), index=False)
    sc.close(); del sc; gc.collect()
SCORES = pd.concat(ALL, ignore_index=True)

Observer: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (13 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  loaded Qwen/Qwen2.5-Coder-7B-Instruct on cuda (torch.bfloat16)
  25/266 tasks (78s)
  50/266 tasks (144s)
  75/266 tasks (220s)
  100/266 tasks (286s)
  125/266 tasks (342s)
  150/266 tasks (394s)
  175/266 tasks (443s)
  200/266 tasks (496s)
  225/266 tasks (554s)
  250/266 tasks (633s)


In [ ]:
# ---- 5  Metrics ----
SIGNALS = {"G": "Sealed gain $G$", "ratio": "Gain per bit $G/S$", "G_un": "Unsealed gain", "low_ppl": "Low perplexity",
           "surprise": "Surprise", "relevance": "Emb.\\ relevance", "judge": "LLM judge", "random": "Random", "U": "Oracle"}
if not CFG["judge"]: SIGNALS.pop("judge")
BASELINES = ["G_un", "low_ppl", "surprise", "relevance", "judge"]
BASELINES = [b for b in BASELINES if b in SIGNALS]

def per_task(df, sig):
    out = []
    for tid, g in df.groupby("task"):
        s, u = g[sig].values.astype(float), g["U"].values
        rho = stats.spearmanr(s, u).correlation if np.std(s) > 0 and np.std(u) > 0 else 0.0
        top = np.argsort(-s, kind="stable")[:3]
        out.append(dict(task=tid, rho=rho, top1_useful=float(g.useful.values[top[0]]),
                        distr_top3=float(np.mean([not g.useful.values[j] for j in top])),
                        acc=g.U_acc.values[top[0]]))
    return pd.DataFrame(out)

def wrong_below(df, sig):
    v = []
    for tid, g in df.groupby("task"):
        code = g[g.type == "CODE"][sig].values; wr = g[g.type == "WRONG"][sig].values
        if len(code) and len(wr): v.append(float(np.all(wr < code[0])))
    return float(np.mean(v))

def dose(df, sig, k, pools, seed=0):
    rng = np.random.default_rng(seed); hits = []
    for tid, g in df.groupby("task"):
        use, ctl = g[g.useful], g[~g.useful]
        for _ in range(pools):
            u = use.iloc[rng.integers(len(use))]; c = ctl.iloc[rng.choice(len(ctl), min(k, len(ctl)), replace=False)]
            s = np.concatenate([[u[sig]], c[sig].values]).astype(float)
            hits.append(float(np.argmax(s + 1e-12 * rng.random(len(s))) == 0))
    return float(np.mean(hits))

RES = {}
for model, df in SCORES.groupby("model"):
    met = {s: per_task(df, s) for s in SIGNALS}
    best = max(BASELINES, key=lambda s: met[s].rho.mean())
    kmax = int((~df[df.task == df.task.iloc[0]].useful).sum())
    r = dict(n_tasks=int(df.task.nunique()), best=best, kmax=kmax,
             p_vs_best=wilcoxon_p(met["G"].rho, met[best].rho),
             signals={s: dict(rho=boot_ci(met[s].rho), top1_useful=float(met[s].top1_useful.mean()),
                              distr_top3=float(met[s].distr_top3.mean()), acc=float(met[s].acc.mean()),
                              wrong_below=wrong_below(df, s), dose=dose(df, s, kmax, CFG["dose_pools"]))
                      for s in SIGNALS},
             types=df.groupby("type")[["S", "G_un", "G", "U"]].mean().round(2).to_dict(orient="index"),
             cp1_pos=float((df.CP1 > 0).mean()), cp1_pos_useful=float((df[df.useful].CP1 > 0).mean()))
    RES[model] = r
    print(f"\n== {model}: {r['n_tasks']} tasks; best baseline {best}; G vs best p={r['p_vs_best']:.3g}")
    for s in SIGNALS:
        m = r["signals"][s]
        print(f"  {s:10s} rho={fmt_ci(m['rho'])}  top1-useful={m['top1_useful']:.2f}  distr@3={m['distr_top3']:.2f}"
              f"  wrong<code={m['wrong_below']:.2f}  1-of-{kmax}={m['dose']:.2f}  dacc={m['acc']:+.3f}")
json.dump(RES, open(os.path.join(OUT, "results.json"), "w"), indent=1, default=float)


== Qwen2.5-7B-Instruct: 266 tasks; best baseline G_un; G vs best p=2.09e-13
  G          rho=0.41 [0.36, 0.45]  top1-useful=0.89  distr@3=0.57  wrong<code=0.24  1-of-8=0.35  dacc=+0.005
  ratio      rho=0.27 [0.23, 0.32]  top1-useful=0.82  distr@3=0.62  wrong<code=0.17  1-of-8=0.30  dacc=+0.008
  G_un       rho=0.24 [0.20, 0.28]  top1-useful=0.17  distr@3=0.58  wrong<code=0.28  1-of-8=0.06  dacc=-0.009
  low_ppl    rho=0.03 [-0.01, 0.07]  top1-useful=0.09  distr@3=0.61  wrong<code=0.92  1-of-8=0.03  dacc=-0.008
  surprise   rho=-0.03 [-0.07, 0.01]  top1-useful=0.00  distr@3=0.90  wrong<code=0.00  1-of-8=0.00  dacc=+0.005
  relevance  rho=0.14 [0.10, 0.18]  top1-useful=0.91  distr@3=0.54  wrong<code=0.18  1-of-8=0.31  dacc=-0.001
  judge      rho=0.10 [0.06, 0.15]  top1-useful=0.99  distr@3=0.31  wrong<code=0.61  1-of-8=0.55  dacc=-0.005
  random     rho=0.03 [-0.01, 0.07]  top1-useful=0.29  distr@3=0.71  wrong<code=0.34  1-of-8=0.12  dacc=+0.003
  U          rho=1.00 [1.00, 1.00]  top

In [ ]:
# ---- 6  Paper inputs: numbers, tables and a figure (nothing typed by hand) ----
def num(v, p=2): return "\\ensuremath{-}" + f"{-v:.{p}f}" if v < 0 else f"{v:.{p}f}"
def pct(v): return f"{100 * v:.0f}\\%"
main = CFG["models"][0].split("/")[-1]
lines = ["% Written by the EvalPlus notebook."]
for model, r in RES.items():
    tag = "ep" if model == main else "ep2"
    s = r["signals"]; lines += [
        f"\\setnum{{{tag}.model}}{{{model}}}", f"\\setnum{{{tag}.ntasks}}{{{r['n_tasks']}}}", f"\\setnum{{{tag}.kmax}}{{{r['kmax']}}}",
        f"\\setnum{{{tag}.best}}{{{SIGNALS[r['best']].replace('$', '')}}}", f"\\setnum{{{tag}.p}}{{{r['p_vs_best']:.2g}}}",
        f"\\setnum{{{tag}.rho.G}}{{{num(s['G']['rho'][0])}}}", f"\\setnum{{{tag}.rho.best}}{{{num(s[r['best']]['rho'][0])}}}",
        f"\\setnum{{{tag}.dose.G}}{{{pct(s['G']['dose'])}}}",
        f"\\setnum{{{tag}.dose.best}}{{{pct(max(s[b]['dose'] for b in BASELINES))}}}",
        f"\\setnum{{{tag}.wrong.G}}{{{pct(s['G']['wrong_below'])}}}", f"\\setnum{{{tag}.distr.G}}{{{pct(s['G']['distr_top3'])}}}",
        f"\\setnum{{{tag}.cp1pos}}{{{pct(r['cp1_pos_useful'])}}}"]
    for ty in ["CODE", "WRONG", "ECHO", "NOISE"]:
        for col in ["S", "G_un", "G", "U"]:
            lines.append(f"\\setnum{{{tag}.{ty.lower()}.{col.replace('_', '')}}}{{{num(r['types'][ty][col], 1)}}}")
open(os.path.join(OUT, "numbers_evalplus.tex"), "w").write("\n".join(lines) + "\n")

r = RES[CFG["models"][0]] if CFG["models"][0] in RES else RES[main]
rows = []
for s, lab in SIGNALS.items():
    m = r["signals"][s]
    dag = "$^\\dagger$" if s in ("G", "ratio") else ""
    rows.append(f"{lab}{dag} & {num(m['rho'][0])} [{num(m['rho'][1])}, {num(m['rho'][2])}] & "
                f"{pct(m['dose'])} & {pct(m['wrong_below'])} & {pct(m['distr_top3'])}\\\\")
    if s == "random": rows.append("\\midrule")
open(os.path.join(OUT, "tab_evalplus.tex"), "w").write(
    "\\begin{tabular}{@{}lcccc@{}}\n\\toprule\nSignal & $\\rho$ with $U$ [95\\% CI] & 1 of " + str(r["kmax"] + 1) +
    " & \\textsc{wrong} $<$ \\textsc{code} & Distr.\\ in top 3\\\\\n\\midrule\n" + "\n".join(rows) + "\n\\bottomrule\n\\end{tabular}\n")

order = USEFUL_TYPES + CONTROL_TYPES
trows = [f"\\textsc{{{ty.lower()}}} & {num(r['types'][ty]['S'], 0)} & {num(r['types'][ty]['G_un'], 1)} & {num(r['types'][ty]['G'], 1)} & {num(r['types'][ty]['U'], 1)}\\\\"
         for ty in order if ty in r["types"]]
trows.insert(len(USEFUL_TYPES), "\\midrule")
open(os.path.join(OUT, "tab_evalplus_types.tex"), "w").write(
    "\\begin{tabular}{@{}lrrrr@{}}\n\\toprule\nType & $S$ & $G_{\\mathrm{un}}$ & $G$ & $U$\\\\\n\\midrule\n" + "\n".join(trows) +
    "\n\\bottomrule\n\\end{tabular}\n")

df = SCORES[SCORES.model == main]; ks = list(range(1, r["kmax"] + 1))
fig, ax = plt.subplots(figsize=(4.6, 2.8))
for s in ["G", "G_un", "low_ppl", "relevance", "judge", "random"]:
    if s not in SIGNALS: continue
    ax.plot(ks, [dose(df, s, k, max(20, CFG["dose_pools"] // 4)) for k in ks], marker="o", ms=3,
            lw=2.2 if s == "G" else 1.2, label=SIGNALS[s].replace("\\", ""))
ax.set(xlabel="number of control documents in the pool", ylabel="P(useful document ranked first)", ylim=(0, 1.02))
ax.legend(fontsize=7, frameon=False); fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_evalplus_dose.pdf")); plt.close(fig)

import shutil
shutil.make_archive(OUT, "zip", OUT)
print("wrote", OUT + ".zip")
if IN_COLAB:
    from google.colab import files; files.download(OUT + ".zip")

wrote /content/evalplus_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>